In [1]:
import jax
import jax.numpy as jnp

In [2]:
# Loss functionZ
@jax.jit
def hybrid_segmentation_loss(
    y_pred: jnp.ndarray, 
    y_true: jnp.ndarray, 
    lambda_weight: float = 0.5, 
    smooth: float = 1e-5
) -> jnp.ndarray:
    """Computes the combined stable BCE and Dice Loss for pixel-wise segmentation."""
    # Clip predictions to prevent log(0) errors in BCE
    y_pred_clipped = jnp.clip(y_pred, 1e-7, 1.0 - 1e-7)
    bce = - (y_true * jnp.log(y_pred_clipped) + (1.0 - y_true) * jnp.log(1.0 - y_pred_clipped))
    bce_loss = jnp.mean(bce)
    
    # Calculate spatial Dice Loss across height and width dimensions
    intersection = jnp.sum(y_pred * y_true, axis=(1, 2))
    denominator = jnp.sum(y_pred, axis=(1, 2)) + jnp.sum(y_true, axis=(1, 2))
    dice_coeff = (2.0 * intersection + smooth) / (denominator + smooth)
    dice_loss = jnp.mean(1.0 - dice_coeff)
    
    # Balance both metrics
    return lambda_weight * dice_loss + (1.0 - lambda_weight) * bce_loss

In [4]:
grad_fn = jax.value_and_grad(hybrid_segmentation_loss)

In [6]:
grad_fn

<function __main__.hybrid_segmentation_loss(y_pred: jax.Array, y_true: jax.Array, lambda_weight: float = 0.5, smooth: float = 1e-05) -> jax.Array>